# 174. Multi-Agent Debate：相关错误、证据共识与停止预算怎样设计？

> **面试问题：多个 LLM/Agent 投票为什么不一定更可靠？怎样处理相关错误、置信校准、证据、Judge 偏差和成本？**

## 先给结论

多数票只有在个体错误不完全相关且答案可规范化时才稳定增益。同一 base、prompt、检索证据和训练数据会制造相关错误，十个克隆不等于十份独立证据。工程上应先独立作答，再匿名交换可核验依据，按校准与相关簇加权，用确定 verifier 优先，并设置停止/预算和单 Agent 基线。

## 推荐回答主线

1. 定义统一答案 parser、agent identity、模型/提示/证据来源与独立首轮，避免先入锚定。
2. 实现多数票和校准置信加权，但把高度相关 agent 聚成簇，限制重复证据权重。
3. debate 交换 claim-evidence/critique，不暴露不必要身份；最终 judge 随机顺序并做位置偏差测试。
4. 用 paired task success、相关切片、校准、成本/延迟和失败联合率决定是否值得启用。

## 教学边界

离线模拟用离散答案、置信和证据哈希，不调用真实 LLM。相关系数与有效样本量是解释近似，不替代真实成对错误矩阵；生产还需处理恶意 agent、隐私、网络故障和动态工具状态。

## 一手资料

- [Multiagent Debate](https://arxiv.org/abs/2305.14325)
- [Self-Consistency](https://arxiv.org/abs/2203.11171)
- [ChatEval](https://arxiv.org/abs/2308.07201)


In [ ]:
import hashlib
import math
from collections import Counter, defaultdict
from dataclasses import dataclass, replace

import numpy as np

# 每个提案记录相关性簇与证据，而不是只有一个裸答案。
@dataclass(frozen=True)
class Proposal:
    agent_id: str
    family: str
    answer: str
    confidence: float
    evidence: tuple[str, ...]
    cost: float

proposals = [
    Proposal("a", "model-x", "42", 0.80, ("calc:6*7",), 1.0),
    Proposal("b", "model-x", "42", 0.85, ("calc:6*7",), 1.0),
    Proposal("c", "model-y", "41", 0.60, ("guess:none",), 1.2),
]

assert len({p.agent_id for p in proposals}) == len(proposals)
assert all(0 < p.confidence < 1 for p in proposals)
assert sum(p.cost for p in proposals) > 0


## 1. 多数票：先规范化答案，并显式处理平票

同义数字、大小写、单位和 JSON 顺序需要任务专用 parser；parser 失败不能随意归到某一答案。多数票只看频数，平票应交 verifier/judge 或返回不确定，不要依赖字典迭代顺序。


In [ ]:
def canonical_answer(answer):
    return " ".join(answer.strip().casefold().split())

def majority_vote(items):
    counts = Counter(canonical_answer(item.answer) for item in items)
    top = counts.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return None, counts
    return top[0][0], counts

# 2:1 多数得到 42；一比一返回平票；大小写/空白被规范化。
winner, counts = majority_vote(proposals)
assert winner == "42" and counts["42"] == 2
assert majority_vote(proposals[:1] + proposals[2:])[0] is None
assert canonical_answer("  YES  ") == "yes"


## 2. 置信加权：先校准，再在 log-odds 空间累加

模型自报 0.9 不一定真有 90% 正确率。可以在冻结校准集把 raw confidence 映射为 empirical accuracy，再用 log-odds 聚合支持度；必须 clip 0/1，且相反答案是多类问题时需分别累计。


In [ ]:
def weighted_support(items, reliability_by_agent, eps=1e-4):
    support = defaultdict(float)
    for item in items:
        calibrated = reliability_by_agent.get(item.agent_id, item.confidence)
        p = min(max(calibrated, eps), 1 - eps)
        support[canonical_answer(item.answer)] += math.log(p / (1 - p))
    return dict(support)

# 两名可靠 agent 对 42 的支持超过单个较弱反对者；支持值有限。
reliability = {"a": 0.75, "b": 0.78, "c": 0.55}
support = weighted_support(proposals, reliability)
assert support["42"] > support["41"]
assert all(math.isfinite(value) for value in support.values())
assert set(support) == {"42", "41"}


## 3. 相关错误：有效独立样本量远小于 Agent 数

若 n 个等质量投票者的错误等相关系数为 ρ，常用设计效应近似给 `n_eff=n/(1+(n-1)ρ)`。ρ=0 才接近 n；ρ=1 时克隆无增益。真实系统应从任务级正确/错误矩阵估计 pairwise correlation。


In [ ]:
def effective_sample_size(n, rho):
    if not (n >= 1 and 0 <= rho <= 1):
        raise ValueError("n/rho 越界")
    return n / (1 + (n - 1) * rho)

# 独立时 n_eff=n；完全相关时为 1；相关性增加使有效样本量单调下降。
assert effective_sample_size(10, 0.0) == 10
assert effective_sample_size(10, 1.0) == 1
assert effective_sample_size(10, 0.7) < effective_sample_size(10, 0.2)


## 4. 相关簇限权：同一 family 的克隆不能无限堆票

可按 base model、prompt、retriever、训练来源或实测错误聚类。下面让同 family 内总权重至多 1，再按成员可靠度分配；它是保守启发式，不能凭 family 标签证明独立。


In [ ]:
def cluster_capped_vote(items, reliability):
    grouped = defaultdict(list)
    for item in items:
        grouped[item.family].append(item)
    score = defaultdict(float)
    for family_items in grouped.values():
        weights = np.array([reliability.get(item.agent_id, 0.5) for item in family_items])
        weights = weights / weights.sum()
        for item, weight in zip(family_items, weights):
            score[canonical_answer(item.answer)] += float(weight)
    return dict(score)

# 复制 model-x agent 不增加该 family 的总票权；每个 family 总贡献为 1。
cluster_score = cluster_capped_vote(proposals, reliability)
cloned = proposals + [Proposal("d", "model-x", "42", 0.99, ("calc:6*7",), 1.0)]
cloned_reliability = {**reliability, "d": 0.99}
cloned_score = cluster_capped_vote(cloned, cloned_reliability)
assert math.isclose(cluster_score["42"], 1.0)
assert math.isclose(cloned_score["42"], 1.0)
assert math.isclose(sum(cluster_score.values()), len({p.family for p in proposals}))


## 5. 证据去重与 verifier 优先：共识不是事实来源

多个 agent 引用同一错误网页仍是一个证据。按内容/来源 hash 去重，优先用计算器、单元测试、数据库约束等确定 verifier。无可验证证据时才让 judge 比较论证，并允许 abstain。


In [ ]:
def evidence_hash(evidence):
    return hashlib.sha256(evidence.encode()).hexdigest()[:16]

def unique_evidence(items):
    by_answer = defaultdict(set)
    for item in items:
        by_answer[canonical_answer(item.answer)].update(evidence_hash(e) for e in item.evidence)
    return {answer: len(values) for answer, values in by_answer.items()}

def arithmetic_verifier(answer):
    return canonical_answer(answer) == str(6 * 7)

# 两个 agent 的同一证据只计一次；确定 verifier 通过 42、拒绝 41。
evidence_counts = unique_evidence(proposals)
assert evidence_counts["42"] == 1
assert arithmetic_verifier("42")
assert not arithmetic_verifier("41")


## 6. Debate 协议：独立首轮、匿名 critique、只允许证据驱动修订

先独立回答可减少锚定；随后交换匿名 claim/evidence，让每个 agent 指出可验证冲突。修订记录 old/new answer、引用证据和原因；若只是“多数都说 X”不算新证据。轮次有上限，防止循环与成本失控。


In [ ]:
@dataclass(frozen=True)
class Revision:
    agent_id: str
    old_answer: str
    new_answer: str
    cited_evidence: str
    reason: str

def accept_revision(revision, known_evidence):
    if revision.old_answer == revision.new_answer:
        return False, "no_change"
    if evidence_hash(revision.cited_evidence) not in known_evidence:
        return False, "unknown_evidence"
    if "大家都" in revision.reason:
        return False, "social_copying"
    return True, "accepted"

# 引用已知算式的修订通过；只随大流或伪造证据被拒绝。
known = {evidence_hash("calc:6*7")}
revision = Revision("c", "41", "42", "calc:6*7", "重新计算得到 42")
assert accept_revision(revision, known)[0]
assert not accept_revision(replace(revision, reason="大家都说 42"), known)[0]
assert accept_revision(replace(revision, cited_evidence="blog:unknown"), known)[1] == "unknown_evidence"


## 7. Judge 偏差与 Byzantine 输入：匿名、换序、截断极端权重

Judge 可能偏爱首位、冗长或特定措辞。评测时随机/交换候选顺序并要求结构化 rubric；对恶意 agent 的极端 confidence 做 calibration/cap，对超长内容和无 provenance 证据拒绝。


In [ ]:
def position_bias_rate(judge, pairs):
    changed = 0
    for left, right in pairs:
        first = judge(left, right)
        swapped = judge(right, left)
        mapped_back = 1 - swapped
        changed += first != mapped_back
    return changed / len(pairs)

# 一个永远选第一项的 judge 在换序测试中暴露 100% 不一致；内容 judge 保持一致。
always_first = lambda left, right: 0
numeric_larger = lambda left, right: 0 if int(left) > int(right) else 1
pairs = [("42", "41"), ("10", "9")]
assert position_bias_rate(always_first, pairs) == 1.0
assert position_bias_rate(numeric_larger, pairs) == 0.0
assert len(pairs) == 2


## 8. 停止与验收：边际收益必须覆盖额外成本和联合失败

当确定 verifier 已通过、答案/证据稳定或预算耗尽就停止。发布时对同一任务成对比较单 Agent 与 debate，报告 success、joint failure、ECE、tokens、tool calls、p95/p99 和每成功任务成本；相关失败 slice 比平均分更重要。


In [ ]:
def should_stop(round_id, max_rounds, verified, answers, budget_left):
    stable = len(answers) >= 2 and answers[-1] == answers[-2]
    if verified:
        return True, "verified"
    if budget_left <= 0 or round_id >= max_rounds:
        return True, "budget"
    if stable:
        return True, "stable"
    return False, "continue"

def paired_report(single_correct, debate_correct, single_cost, debate_cost):
    single_correct, debate_correct = np.asarray(single_correct), np.asarray(debate_correct)
    return {
        "single_accuracy": float(single_correct.mean()),
        "debate_accuracy": float(debate_correct.mean()),
        "fixed_failures": int(((single_correct == 0) & (debate_correct == 1)).sum()),
        "regressions": int(((single_correct == 1) & (debate_correct == 0)).sum()),
        "extra_cost": float(debate_cost - single_cost),
    }

# verifier 立即终止；稳定答案终止；成对报告能同时看到修复和回归。
assert should_stop(0, 3, True, ["42"], 10)[1] == "verified"
assert should_stop(1, 3, False, ["42", "42"], 10)[1] == "stable"
report = paired_report([1, 0, 1, 0], [1, 1, 0, 0], 4.0, 11.0)
assert report["fixed_failures"] == 1 and report["regressions"] == 1


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
